In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name
)


ROOT="/Volumes/capstone4_retail/retail_orders/ext_capstone4_vol"

PRIMARY_PATH = ROOT + "/primary_events"

STATUS_PATH = ROOT + "/status_updates"

SECONDARY_PATH = ROOT + "/secondary_events"

CHECKPOINT_ROOT = ROOT + "/checkpoints"

SCHEMA_ROOT = ROOT + "/schema_tracking"

CATALOG = "retail_capstone"

SCHEMA = "retail"

In [0]:
def ingest_with_autoloader(
    source_path,
    table_name,
    checkpoint_name,
    schema_name
):

    checkpoint_path = (
        CHECKPOINT_ROOT + "/" + checkpoint_name
    )

    schema_path = (
        SCHEMA_ROOT + "/" + schema_name
    )

    print("Starting ingestion:", table_name)

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )
        .option(
            "cloudFiles.inferColumnTypes",
            "false"
        )
        .option("header", "true")
        .load(source_path)
    )

    bronze_df = (
        df
        .withColumn(
            "source_file",
            input_file_name()
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
    )

    query = (
        bronze_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(
            CATALOG + "." + SCHEMA + "." + table_name
        )
    )

    query.awaitTermination()

    print("Completed ingestion:", table_name)

In [0]:
from pyspark.sql.functions import col

input_file_name = lambda: col("_metadata.file_path")

CATALOG = "capstone4_retail"
SCHEMA = "retail_orders"

ingest_with_autoloader(
    PRIMARY_PATH,
    "bronze_sale_events",
    "primary",
    "primary"
)

In [0]:
ingest_with_autoloader(
    STATUS_PATH,
    "bronze_status_cdc",
    "status",
    "status"
)

In [0]:
ingest_with_autoloader(
    SECONDARY_PATH,
    "bronze_secondary_events",
    "secondary",
    "secondary"
)

In [0]:
%sql
SELECT
    'bronze_sale_events' AS table_name,
    COUNT(*) AS total_records
FROM capstone4_retail.retail_orders.bronze_sale_events

UNION ALL

SELECT
    'bronze_status_cdc',
    COUNT(*)
FROM capstone4_retail.retail_orders.bronze_status_cdc

UNION ALL

SELECT
    'bronze_secondary_events',
    COUNT(*)
FROM capstone4_retail.retail_orders.bronze_secondary_events
UNION ALL

SELECT
    'bronze_store_id',
    COUNT(*)
FROM capstone4_retail.retail_orders.bronze_store_id

UNION ALL

SELECT
    'bronze_product_id',
    COUNT(*)
FROM capstone4_retail.retail_orders.bronze_product_id;